In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ---------- settings ----------
outdir = Path("plots")
outdir.mkdir(exist_ok=True)

infile = "./haplotypes_min10_lc0.01.csv"

# ---------- load ----------
df = pd.read_csv(infile)

# Accuracy: REF correctly assigned to H2
df["accurate"] = df["REF"].astype(str) == df["H2"].astype(str)

# ---------- summarise ----------
stage_summary = (
    df.groupby("Stage", dropna=False)["accurate"]
      .agg(
          n_pairs="size",
          n_correct="sum",
          accuracy="mean"
      )
      .reset_index()
)

stage_summary["Stage"] = stage_summary["Stage"].astype(str)
stage_summary["label"] = "Stage " + stage_summary["Stage"]

overall = pd.DataFrame({
    "Stage": ["Overall"],
    "label": ["Overall"],
    "n_pairs": [len(df)],
    "n_correct": [df["accurate"].sum()],
    "accuracy": [df["accurate"].mean()],
})

summary = pd.concat([stage_summary, overall], ignore_index=True)
summary["n_incorrect"] = summary["n_pairs"] - summary["n_correct"]
summary["pct_accuracy"] = summary["accuracy"] * 100
summary["pct_incorrect"] = 100 - summary["pct_accuracy"]

summary["plot_order"] = summary["Stage"].apply(
    lambda x: 999 if x == "Overall" else float(x)
)
summary = summary.sort_values("plot_order").reset_index(drop=True)

# # ---------- print summary ----------
# print("\nPhasing accuracy summary")
# print("=" * 70)
# print(
#     summary[
#         ["label", "n_pairs", "n_correct", "n_incorrect", "pct_accuracy"]
#     ].to_string(
#         index=False,
#         formatters={
#             "pct_accuracy": lambda x: f"{x:.2f}%"
#         }
#     )
# )

# summary_out = outdir / "phasing_accuracy_summary.tsv"
# summary[
#     ["label", "n_pairs", "n_correct", "n_incorrect", "pct_accuracy"]
# ].to_csv(summary_out, sep="\t", index=False)

# print(f"\nWrote summary table: {summary_out}")

# # ---------- style ----------
# plt.rcParams.update({
#     "font.family": "DejaVu Sans",
#     "font.size": 10,
#     "axes.titlesize": 13,
#     "axes.labelsize": 11,
#     "xtick.labelsize": 10,
#     "ytick.labelsize": 10,
#     "legend.fontsize": 9,
#     "axes.linewidth": 0.8,
#     "pdf.fonttype": 42,
#     "ps.fonttype": 42,
# })


def save_all(fig, stem):
    for ext in ["png", "pdf", "svg", "eps"]:
        fig.savefig(
            outdir / f"{stem}.{ext}",
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )

    print(f"Wrote: {outdir / stem}.png/pdf/svg/eps")


# # ============================================================
# # Plot 1: Main accuracy lollipop plot
# # ============================================================
# fig, ax = plt.subplots(figsize=(5.4, 3.8))

# x = np.arange(len(summary))
# y = summary["pct_accuracy"].values

# ax.vlines(
#     x=x,
#     ymin=50,
#     ymax=y,
#     linewidth=2,
#     alpha=0.8,
# )

# ax.scatter(
#     x,
#     y,
#     s=95,
#     edgecolor="black",
#     linewidth=0.9,
#     zorder=3,
# )

# ax.axhline(
#     50,
#     linestyle="--",
#     linewidth=1,
#     color="grey",
#     alpha=0.8,
# )

# for i, row in summary.iterrows():
#     ax.text(
#         i,
#         row["pct_accuracy"] + 1.8,
#         f'{row["pct_accuracy"]:.1f}%',
#         ha="center",
#         va="bottom",
#         fontsize=10,
#         fontweight="bold",
#     )
#     ax.text(
#         i,
#         4,
#         f'n={int(row["n_pairs"])}',
#         ha="center",
#         va="bottom",
#         fontsize=9,
#         color="dimgray",
#     )

# ax.set_xticks(x)
# ax.set_xticklabels(summary["label"])
# ax.set_ylim(0, 105)
# ax.set_ylabel("Accuracy (%)")
# ax.set_title("Phasing accuracy by stage")
# ax.text(
#     -0.45,
#     51.5,
#     "Random expectation",
#     fontsize=8.5,
#     color="dimgray",
#     va="bottom",
# )

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# ax.grid(axis="y", alpha=0.22)
# ax.set_axisbelow(True)

# fig.tight_layout()
# save_all(fig, "phasing_accuracy_lollipop_by_stage")
# plt.show()


# # ============================================================
# # Plot 2: 100% stacked accuracy/error proportion plot
# # ============================================================
# fig, ax = plt.subplots(figsize=(5.4, 3.8))

# x = np.arange(len(summary))

# ax.bar(
#     x,
#     summary["pct_accuracy"],
#     width=0.65,
#     label="Correct",
#     edgecolor="black",
#     linewidth=0.8,
# )

# ax.bar(
#     x,
#     summary["pct_incorrect"],
#     bottom=summary["pct_accuracy"],
#     width=0.65,
#     label="Incorrect",
#     edgecolor="black",
#     linewidth=0.8,
# )

# for i, row in summary.iterrows():
#     ax.text(
#         i,
#         row["pct_accuracy"] / 2,
#         f'{row["pct_accuracy"]:.1f}%',
#         ha="center",
#         va="center",
#         fontsize=9,
#         fontweight="bold",
#     )
#     ax.text(
#         i,
#         102,
#         f'n={int(row["n_pairs"])}',
#         ha="center",
#         va="bottom",
#         fontsize=9,
#         color="dimgray",
#     )

# ax.set_xticks(x)
# ax.set_xticklabels(summary["label"])
# ax.set_ylim(0, 110)
# ax.set_ylabel("SNPs (%)")
# ax.set_title("Correct vs incorrect SNP assignment")
# ax.legend(frameon=False, loc="upper right")

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# ax.grid(axis="y", alpha=0.22)
# ax.set_axisbelow(True)

# fig.tight_layout()
# save_all(fig, "phasing_accuracy_stacked_percent_by_stage")
# plt.show()


# # ============================================================
# # Plot 3: Number of SNPs per stage / overall
# # ============================================================
# fig, ax = plt.subplots(figsize=(5.4, 3.8))

# bars = ax.bar(
#     x,
#     summary["n_pairs"],
#     width=0.65,
#     edgecolor="black",
#     linewidth=0.8,
# )

# for bar, (_, row) in zip(bars, summary.iterrows()):
#     ax.text(
#         bar.get_x() + bar.get_width() / 2,
#         row["n_pairs"] + max(summary["n_pairs"]) * 0.02,
#         f'{int(row["n_pairs"]):,}',
#         ha="center",
#         va="bottom",
#         fontsize=10,
#         fontweight="bold",
#     )

# ax.set_xticks(x)
# ax.set_xticklabels(summary["label"])
# ax.set_ylabel("Number of SNPs")
# ax.set_title("SNPs contributing to accuracy estimate")

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)
# ax.grid(axis="y", alpha=0.22)
# ax.set_axisbelow(True)

# fig.tight_layout()
# save_all(fig, "phasing_accuracy_n_snps_by_stage")
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

In [ ]:
# ============================================================
# 40 mm × 40 mm lollipop plot: phasing accuracy by stage
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 40
FIG_HEIGHT_MM = 40
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

# ---------- publication-style font settings for small panel ----------
plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.5,
    "axes.labelsize": 6.0,
    "axes.titlesize": 6.2,
    "xtick.labelsize": 5.4,
    "ytick.labelsize": 5.4,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})

# ---------- order: Stage 1, Stage 2, Overall ----------
def clean_stage_label(x):
    if str(x) == "Overall":
        return "Overall"
    return f"Stage {int(float(x))}"


summary["label"] = summary["Stage"].apply(clean_stage_label)

summary["plot_order"] = summary["Stage"].apply(
    lambda x: 999 if str(x) == "Overall" else float(x)
)

summary = summary.sort_values("plot_order").reset_index(drop=True)

# ---------- colours ----------
color_map = {
    "Stage 1": "#2F6F9F",
    "Stage 2": "#E76F51",
    "Overall": "#6D597A",
}

# ============================================================
# Plot: compact accuracy lollipop plot
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE)

x = np.arange(len(summary)) * 0.62
y = summary["pct_accuracy"].values
point_colors = summary["label"].map(color_map).tolist()

# ---------- lollipops ----------
for xi, yi, c in zip(x, y, point_colors):
    ax.vlines(
        x=xi,
        ymin=50,
        ymax=yi,
        linewidth=1.25,
        alpha=0.95,
        color=c,
        zorder=2,
    )

    ax.scatter(
        xi,
        yi,
        s=34,
        color=c,
        edgecolor="black",
        linewidth=0.45,
        zorder=3,
    )

# ---------- random expectation ----------
ax.axhline(
    50,
    linestyle="--",
    linewidth=0.65,
    color="red",
    alpha=0.8,
)

ax.text(
    x.min() - 0.5,
    50,
    "Random\nExpectation",
    fontsize=4.7,
    color="red",
    ha="center",
    va="bottom",
    style="italic",
)

# ---------- value annotations ----------
for i, row in summary.iterrows():
    ax.annotate(
        f'{row["pct_accuracy"]:.2f}%',
        xy=(x[i], row["pct_accuracy"]),
        xytext=(0, 3.2),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=4.9,
        fontweight="bold",
    )

    ax.annotate(
        f'{int(row["n_pairs"]):,}',
        xy=(x[i], 0),
        xytext=(0, 3.2),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=4.3,
        color="dimgray",
        style="italic",
    )

# ---------- axes ----------
ax.set_xticks(x)
ax.set_xticklabels(summary["label"], rotation=45, ha="right", rotation_mode="anchor")
ax.set_xlim(x.min() - 1, x.max() + 0.28)
ax.set_ylim(0, 108)

ax.set_ylabel("Accuracy (%)", labelpad=1.5)
# ax.set_title("Phasing accuracy", pad=3)

# ---------- SNP count label ----------
ax.text(
    x.min() - 0.20,
    0.045,
    "n SNPs:",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=4.3,
    color="dimgray",
    style="italic",
)

# ---------- styling ----------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(axis="y", alpha=0.22, linewidth=0.45)
ax.set_axisbelow(True)

ax.tick_params(axis="x", pad=1.5)
ax.tick_params(axis="y", pad=1.2)

fig.subplots_adjust(
    left=0.23,
    right=0.98,
    top=0.87,
    bottom=0.24,
)

save_all(fig, "phasing_accuracy_lollipop_by_stage_40x40mm")
plt.show()

In [ ]:
# Order: Stage 1, Stage 2, Overall
def clean_stage_label(x):
    if str(x) == "Overall":
        return "Overall"
    return f"Stage {int(float(x))}"

summary["label"] = summary["Stage"].apply(clean_stage_label)

summary["plot_order"] = summary["Stage"].apply(
    lambda x: 999 if str(x) == "Overall" else float(x)
)
summary = summary.sort_values("plot_order").reset_index(drop=True)

# Same colours as the stacked bar
color_map = {
    "Stage 1": "#2F6F9F",   # muted blue
    "Stage 2": "#E76F51",   # coral
    "Overall": "#6D597A",   # muted purple
}

# ============================================================
# Plot 1: Main accuracy lollipop plot
# ============================================================
fig, ax = plt.subplots(figsize=(5.2, 4.0))

x = np.arange(len(summary)) * 0.72
y = summary["pct_accuracy"].values
point_colors = summary["label"].map(color_map).tolist()

# Draw each lollipop separately so each one has its own colour
for xi, yi, c in zip(x, y, point_colors):
    ax.vlines(
        x=xi,
        ymin=50,
        ymax=yi,
        linewidth=2.5,
        alpha=0.9,
        color=c,
        zorder=2,
    )
    ax.scatter(
        xi,
        yi,
        s=130,
        color=c,
        edgecolor="black",
        linewidth=1.1,
        zorder=3,
    )

# Random expectation line/label in the same accent purple
ax.axhline(50, linestyle="--", linewidth=1.1, color="red", alpha=0.85)

for i, row in summary.iterrows():
    ax.annotate(
        f'{row["pct_accuracy"]:.2f}%',
        xy=(x[i], row["pct_accuracy"]),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10.5,
        fontweight="bold",
    )

    ax.annotate(
        f'{int(row["n_pairs"]):,}',
        xy=(x[i], 0),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8.5,
        color="dimgray",
        style="italic",
    )

ax.text(
    -0.3,
    60.0,
    "Random\nexpectation",
    fontsize=9,
    color="red",
    ha="center",
    va="top",
    style="italic",
)

ax.set_xticks(x)
ax.set_xticklabels(summary["label"])

# Extra gap between y-axis and first lollipop
ax.set_xlim(x.min() - 0.56, x.max() + 0.28)

ax.set_ylim(0, 108)
ax.set_ylabel("Accuracy (%)")
ax.set_title("ChrX Phasing accuracy by algorithm stage", pad=12)

# Single left-side label for the SNP counts
ax.text(
    x.min() - 0.18,
    0.03,
    "Number\nof SNPs:",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="bottom",
    fontsize=8.5,
    color="dimgray",
    style="italic",
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)

fig.tight_layout()
save_all(fig, "phasing_accuracy_lollipop_by_stage")
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

# ============================================================
# One vertical stacked bar: SNP counts + proportions by stage
# ============================================================

stage_counts = summary[summary["Stage"].astype(str) != "Overall"].copy()

stage_counts["label"] = stage_counts["Stage"].apply(
    lambda x: f"Stage {int(float(x))}"
)

stage_counts["plot_order"] = stage_counts["Stage"].astype(float)
stage_counts = stage_counts.sort_values("plot_order").reset_index(drop=True)

total_snps = int(stage_counts["n_pairs"].sum())
stage_counts["proportion"] = stage_counts["n_pairs"] / total_snps

# Coordinated colours
stage_colors = ["#2F6F9F", "#E76F51"]   # muted blue, coral
overall_color = "#6D597A"               # muted purple for bracket/overall label

fig, ax = plt.subplots(figsize=(3.6, 4.0))

bottom = 0
bar_x = 0
bar_width = 0.58

for i, (_, row) in enumerate(stage_counts.iterrows()):
    height = int(row["n_pairs"])
    prop = row["proportion"] * 100

    ax.bar(
        bar_x,
        height,
        bottom=bottom,
        width=bar_width,
        color=stage_colors[i],
        edgecolor="black",
        linewidth=0.9,
        zorder=3,
    )

    ax.text(
        bar_x,
        bottom + height / 2,
        f'{row["label"]}\n{height:,} SNPs\n({prop:.1f}%)',
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        linespacing=1.2,
        zorder=4,
    )

    bottom += height

# Primary y-axis: SNP counts
ax.set_ylim(0, 5000)
ax.set_ylabel("Number of SNPs")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}"))

# Secondary y-axis: proportion
def count_to_prop(y):
    return y / total_snps

def prop_to_count(p):
    return p * total_snps

secax = ax.secondary_yaxis(
    "right",
    functions=(count_to_prop, prop_to_count)
)

secax.set_ylabel(
    "Proportion of SNPs",
    rotation=270,
    labelpad=18,
)

secax.set_yticks([0, 0.25, 0.50, 0.75, 1.00])
secax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"])

# Make right axis match overall/bracket colour
# secax.yaxis.label.set_color(overall_color)
# secax.tick_params(axis="y", colors=overall_color)
# secax.spines["right"].set_color(overall_color)

# More room on the right between the bar and the axis
ax.set_xlim(-0.32, 0.62)

# Remove x-axis ticks and tick labels
ax.set_xticks([])
ax.tick_params(axis="x", which="both", bottom=False, top=False, labelbottom=False)

# Dashed line at the overall total

# ------------------------------------------------------------
# Square bracket on the right showing total bar height
# ------------------------------------------------------------
bar_right = bar_x + bar_width / 2
bracket_x = bar_right + 0.12
bracket_tick = 0.05

# vertical bracket line
ax.plot(
    [bracket_x, bracket_x],
    [0, total_snps],
    color=overall_color,
    linewidth=1.5,
    clip_on=False,
    zorder=5,
)

# bottom bracket tick
ax.plot(
    [bracket_x - bracket_tick, bracket_x],
    [0, 0],
    color=overall_color,
    linewidth=1.5,
    clip_on=False,
    zorder=5,
)

# top bracket tick
ax.plot(
    [bracket_x - bracket_tick, bracket_x],
    [total_snps, total_snps],
    color=overall_color,
    linewidth=1.5,
    clip_on=False,
    zorder=5,
)

# Overall label, rotated the other way
ax.text(
    bracket_x + 0.08,
    total_snps / 2,
    f"Overall\n{total_snps:,} SNPs",
    ha="center",
    va="center",
    fontsize=8.8,
    color=overall_color,
    fontweight="bold",
    rotation=270,
)

# Title
ax.set_title("Expressed SNPs by scDaisychain Stage", pad=12)

# Styling
ax.spines["top"].set_visible(False)
secax.spines["top"].set_visible(False)
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)

fig.tight_layout()
save_all(fig, "phasing_accuracy_snp_counts_vertical_stacked")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

run_dir = Path(".")
gt_dir = run_dir / "ground_truth_split_bams_count/matrices"
pred_dir = run_dir/"daisychain_split_bams_count/matrices"  # change if needed

from pathlib import Path

outdir = Path("/home/913/dk4874/scratch/gdata/scDaisychain_paper/mouse/scripts/analysis/plots")
outdir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def savefig(name):
    plt.tight_layout()
    for ext in ["pdf", "svg", "eps", "png"]:
        plt.savefig(outdir / f"{name}.{ext}", bbox_inches="tight", dpi=300)
    plt.show()

def load_mats(d):
    return {x: pd.read_csv(d / f"{x}.csv", index_col=0) for x in ["X1", "X2", "Xa", "Xi", "amb"]}

gt = load_mats(gt_dir)
pred_raw = load_mats(pred_dir)

common_genes = gt["Xa"].index.intersection(pred_raw["Xa"].index)
common_cells = gt["Xa"].columns.intersection(pred_raw["Xa"].columns)

gt = {k: v.loc[common_genes, common_cells] for k, v in gt.items()}
pred_raw = {k: v.loc[common_genes, common_cells] for k, v in pred_raw.items()}

# X1/X2 fix for raw haplotype matrices only
pred = pred_raw.copy()
pred["X1_fixed"] = pred_raw["X2"]
pred["X2_fixed"] = pred_raw["X1"]

def corr_text(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return "r = NA\nρ = NA"
    r = pearsonr(x[mask], y[mask])[0]
    rho = spearmanr(x[mask], y[mask])[0]
    return f"r = {r:.3f}\nρ = {rho:.3f}"

def add_identity(ax, x, y):
    lim = max(np.nanmax(x), np.nanmax(y))
    ax.plot([0, lim], [0, lim], "--", color="black", linewidth=0.8, alpha=0.7)
    ax.set_xlim(-0.02 * lim, 1.03 * lim)
    ax.set_ylim(-0.02 * lim, 1.03 * lim)

def panel_scatter(ax, x, y, c=None, xlabel="", ylabel="", title="", size=12, cmap="viridis"):
    if c is None:
        ax.scatter(x, y, s=size, alpha=0.65, linewidths=0)
        sc = None
    else:
        sc = ax.scatter(x, y, c=c, s=size, alpha=0.75, linewidths=0, cmap=cmap)
    add_identity(ax, np.asarray(x), np.asarray(y))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.text(
        0.04, 0.96, corr_text(np.asarray(x), np.asarray(y)),
        transform=ax.transAxes,
        va="top", ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8),
    )
    return sc

# ---------------- matrix total panels ----------------

fig, axes = plt.subplots(2, 3, figsize=(8.2, 5.2))
axes = axes.ravel()

plot_specs = [
    ("X1", "X1_fixed"),
    ("X2", "X2_fixed"),
    ("Xa", "Xa"),
    ("Xi", "Xi"),
    ("amb", "amb"),
]

for ax, (mat, pred_mat) in zip(axes, plot_specs):
    x = gt[mat].sum(axis=0)
    y = pred[pred_mat].sum(axis=0)
    panel_scatter(
        ax, x, y,
        xlabel=f"Ground truth {mat}",
        ylabel=f"Daisychain {mat}",
        title=f"{mat} per cell",
        size=8,
    )

axes[-1].axis("off")
fig.suptitle("Per-cell matrix totals", y=1.02, fontsize=12)
savefig("panel_per_cell_matrix_totals")

fig, axes = plt.subplots(2, 3, figsize=(8.2, 5.2))
axes = axes.ravel()

for ax, (mat, pred_mat) in zip(axes, plot_specs):
    x = gt[mat].sum(axis=1)
    y = pred[pred_mat].sum(axis=1)
    panel_scatter(
        ax, x, y,
        xlabel=f"Ground truth {mat}",
        ylabel=f"Daisychain {mat}",
        title=f"{mat} per gene",
        size=18,
    )

axes[-1].axis("off")
fig.suptitle("Per-gene matrix totals", y=1.02, fontsize=12)
savefig("panel_per_gene_matrix_totals")

# ---------------- Xi ratio ----------------

def xi_ratio_and_total(xa, xi, axis):
    xa_sum = xa.sum(axis=axis)
    xi_sum = xi.sum(axis=axis)
    total = xa_sum + xi_sum
    ratio = xi_sum / total.replace(0, np.nan)
    return ratio, total

gt_cell_ratio, gt_cell_total = xi_ratio_and_total(gt["Xa"], gt["Xi"], axis=0)
pred_cell_ratio, pred_cell_total = xi_ratio_and_total(pred["Xa"], pred["Xi"], axis=0)

cell_cmp = pd.DataFrame({
    "groundtruth_xi_ratio": gt_cell_ratio,
    "daisychain_xi_ratio": pred_cell_ratio,
    "groundtruth_classified_reads": gt_cell_total,
    "daisychain_classified_reads": pred_cell_total,
}).dropna()
cell_cmp["classified_reads"] = cell_cmp[["groundtruth_classified_reads", "daisychain_classified_reads"]].mean(axis=1)
cell_cmp["diff"] = cell_cmp["daisychain_xi_ratio"] - cell_cmp["groundtruth_xi_ratio"]
cell_cmp["abs_diff"] = cell_cmp["diff"].abs()

gt_gene_ratio, gt_gene_total = xi_ratio_and_total(gt["Xa"], gt["Xi"], axis=1)
pred_gene_ratio, pred_gene_total = xi_ratio_and_total(pred["Xa"], pred["Xi"], axis=1)

gene_cmp = pd.DataFrame({
    "groundtruth_xi_ratio": gt_gene_ratio,
    "daisychain_xi_ratio": pred_gene_ratio,
    "groundtruth_classified_reads": gt_gene_total,
    "daisychain_classified_reads": pred_gene_total,
}).dropna()
gene_cmp["classified_reads"] = gene_cmp[["groundtruth_classified_reads", "daisychain_classified_reads"]].mean(axis=1)
gene_cmp["diff"] = gene_cmp["daisychain_xi_ratio"] - gene_cmp["groundtruth_xi_ratio"]
gene_cmp["abs_diff"] = gene_cmp["diff"].abs()

cell_cmp.to_csv(outdir / "xi_ratio_per_cell.tsv", sep="\t")
gene_cmp.to_csv(outdir / "xi_ratio_per_gene.tsv", sep="\t")

# Main Xi-ratio panel
fig = plt.figure(figsize=(8.5, 3.8))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
cax = fig.add_subplot(gs[0, 2])

vmin = min(cell_cmp["classified_reads"].min(), gene_cmp["classified_reads"].min())
vmax = max(cell_cmp["classified_reads"].max(), gene_cmp["classified_reads"].max())

sc1 = ax1.scatter(
    cell_cmp["groundtruth_xi_ratio"],
    cell_cmp["daisychain_xi_ratio"],
    c=cell_cmp["classified_reads"],
    s=9,
    alpha=0.75,
    linewidths=0,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
ax1.plot([0, 1], [0, 1], "--", color="black", linewidth=0.8)
ax1.set_xlim(-0.03, 1.03)
ax1.set_ylim(-0.03, 1.03)
ax1.set_xlabel("Ground truth Xi ratio")
ax1.set_ylabel("Daisychain Xi ratio")
ax1.set_title("Per cell")
ax1.text(
    0.04, 0.96,
    corr_text(cell_cmp["groundtruth_xi_ratio"].values, cell_cmp["daisychain_xi_ratio"].values),
    transform=ax1.transAxes,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8),
)

sc2 = ax2.scatter(
    gene_cmp["groundtruth_xi_ratio"],
    gene_cmp["daisychain_xi_ratio"],
    c=gene_cmp["classified_reads"],
    s=26,
    alpha=0.85,
    linewidths=0,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
ax2.plot([0, 1], [0, 1], "--", color="black", linewidth=0.8)
ax2.set_xlim(-0.03, 1.03)
ax2.set_ylim(-0.03, 1.03)
ax2.set_xlabel("Ground truth Xi ratio")
ax2.set_ylabel("Daisychain Xi ratio")
ax2.set_title("Per gene")
ax2.text(
    0.04, 0.96,
    corr_text(gene_cmp["groundtruth_xi_ratio"].values, gene_cmp["daisychain_xi_ratio"].values),
    transform=ax2.transAxes,
    va="top",
    ha="left",
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8),
)

cb = fig.colorbar(sc2, cax=cax)
cb.set_label("Total classified Xa + Xi reads")

fig.suptitle("Xi-ratio benchmarking", y=1.04, fontsize=12)
savefig("panel_xi_ratio_cell_gene_viridis")

# Xi-ratio error panel
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))

axes[0].hist(cell_cmp["diff"], bins=50, color="#4C78A8", edgecolor="black", linewidth=0.25)
axes[0].axvline(0, linestyle="--", color="black", linewidth=0.8)
axes[0].set_xlabel("Daisychain - ground truth Xi ratio")
axes[0].set_ylabel("Cells")
axes[0].set_title("Per-cell error")

axes[1].hist(gene_cmp["diff"], bins=40, color="#4C78A8", edgecolor="black", linewidth=0.25)
axes[1].axvline(0, linestyle="--", color="black", linewidth=0.8)
axes[1].set_xlabel("Daisychain - ground truth Xi ratio")
axes[1].set_ylabel("Genes")
axes[1].set_title("Per-gene error")

fig.suptitle("Xi-ratio error distribution", y=1.05, fontsize=12)
savefig("panel_xi_ratio_error_histograms")

# Top gene outlier panel
top = gene_cmp.sort_values("abs_diff", ascending=False).head(15)
top.to_csv(outdir / "top_gene_xi_ratio_outliers.tsv", sep="\t")

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))

axes[0].scatter(
    gene_cmp["groundtruth_xi_ratio"],
    gene_cmp["daisychain_xi_ratio"],
    c=gene_cmp["classified_reads"],
    s=28,
    alpha=0.85,
    linewidths=0,
    cmap="viridis",
)
axes[0].plot([0, 1], [0, 1], "--", color="black", linewidth=0.8)
axes[0].set_xlim(-0.03, 1.03)
axes[0].set_ylim(-0.03, 1.03)
axes[0].set_xlabel("Ground truth Xi ratio")
axes[0].set_ylabel("Daisychain Xi ratio")
axes[0].set_title("Gene Xi ratios")

for gene in top.index[:8]:
    r = gene_cmp.loc[gene]
    axes[0].text(
        r["groundtruth_xi_ratio"] + 0.01,
        r["daisychain_xi_ratio"] + 0.01,
        gene,
        fontsize=7,
    )

ypos = np.arange(len(top))[::-1]
axes[1].barh(ypos, top["abs_diff"].values, color="#4C78A8")
axes[1].set_yticks(ypos)
axes[1].set_yticklabels(top.index)
axes[1].set_xlabel("Absolute Xi-ratio error")
axes[1].set_title("Top gene outliers")

fig.suptitle("Gene-level Xi-ratio outliers", y=1.05, fontsize=12)
savefig("panel_gene_xi_ratio_outliers")

# Summary table
summary = {
    "n_common_genes": len(common_genes),
    "n_common_cells": len(common_cells),
    "cell_pearson": pearsonr(cell_cmp["groundtruth_xi_ratio"], cell_cmp["daisychain_xi_ratio"])[0],
    "cell_spearman": spearmanr(cell_cmp["groundtruth_xi_ratio"], cell_cmp["daisychain_xi_ratio"])[0],
    "cell_mean_abs_error": cell_cmp["abs_diff"].mean(),
    "cell_median_abs_error": cell_cmp["abs_diff"].median(),
    "gene_pearson": pearsonr(gene_cmp["groundtruth_xi_ratio"], gene_cmp["daisychain_xi_ratio"])[0],
    "gene_spearman": spearmanr(gene_cmp["groundtruth_xi_ratio"], gene_cmp["daisychain_xi_ratio"])[0],
    "gene_mean_abs_error": gene_cmp["abs_diff"].mean(),
    "gene_median_abs_error": gene_cmp["abs_diff"].median(),
}
pd.DataFrame([summary]).to_csv(outdir / "benchmarking_summary.tsv", sep="\t", index=False)
display(pd.DataFrame([summary]))

print("Saved editable plots to:", outdir)

In [ ]:
# ---------------- matrix total panels: per cell, no amb
# Compact 50 mm × 50 mm version, transparent background
# ----------------

import numpy as np
import matplotlib.pyplot as plt

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 50
FIG_HEIGHT_MM = 50
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.4,
    "axes.labelsize": 5.6,
    "axes.titlesize": 6.0,
    "xtick.labelsize": 4.8,
    "ytick.labelsize": 4.8,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 1.8,
    "ytick.major.size": 1.8,
})

plot_specs = [
    ("X1", "X1_fixed"),
    ("X2", "X2_fixed"),
    ("Xa", "Xa"),
    ("Xi", "Xi"),
]

point_color = "#4C78A8"


def panel_scatter_pretty(
    ax,
    x,
    y,
    title="",
    show_xlabel=False,
    show_ylabel=False,
):
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) == 0:
        ax.text(
            0.5,
            0.5,
            "No data",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=5,
        )
        return

    lim = max(np.nanmax(x), np.nanmax(y))
    pad = lim * 0.05 if lim > 0 else 1

    ax.scatter(
        x,
        y,
        s=4.5,
        alpha=0.50,
        color=point_color,
        linewidths=0,
        rasterized=True,
    )

    ax.plot(
        [0, lim],
        [0, lim],
        linestyle="--",
        color="black",
        linewidth=0.45,
        alpha=0.55,
        zorder=0,
    )

    ax.set_xlim(-pad, lim + pad)
    ax.set_ylim(-pad, lim + pad)
    ax.set_aspect("equal", adjustable="box")

    ax.set_title(title, fontweight="bold", pad=2)

    if show_xlabel:
        ax.set_xlabel("Ground truth", labelpad=1.0)
    else:
        ax.set_xlabel("")

    if show_ylabel:
        ax.set_ylabel("Daisychain", labelpad=1.0)
    else:
        ax.set_ylabel("")

    ax.grid(alpha=0.18, linewidth=0.35)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.text(
        0.05,
        0.95,
        corr_text(x, y),
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=4.4,
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="0.85",
            alpha=0.88,
            linewidth=0.35,
        ),
    )


fig, axes = plt.subplots(
    2,
    2,
    figsize=FIGSIZE,
)

axes = axes.ravel()

for idx, (ax, (mat, pred_mat)) in enumerate(zip(axes, plot_specs)):
    x = gt[mat].sum(axis=0)
    y = pred[pred_mat].sum(axis=0)

    panel_scatter_pretty(
        ax,
        x,
        y,
        title=f"{mat}",
        show_xlabel=idx in [2, 3],
        show_ylabel=idx in [0, 2],
    )

# Transparent background
fig.patch.set_alpha(0)
for ax in fig.axes:
    ax.patch.set_alpha(0)

fig.suptitle(
    "Per-cell matrix totals",
    y=0.995,
    fontsize=6.4,
    fontweight="bold",
)

fig.subplots_adjust(
    left=0.17,
    right=0.98,
    bottom=0.13,
    top=0.90,
    wspace=0.38,
    hspace=0.42,
)

savefig("panel_per_cell_matrix_totals_no_amb_pretty_50x50mm")
plt.show()

In [ ]:
# ============================================================
# Xi ratio per cell and per gene — min 10 counts in BOTH methods
# Compact 50 mm × 50 mm version, transparent background
# No overall plot title
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

min_counts = 10
point_color = "#4C78A8"

# ---------- figure size ----------
MM_PER_INCH = 25.4
FIG_WIDTH_MM = 50
FIG_HEIGHT_MM = 50
FIGSIZE = (FIG_WIDTH_MM / MM_PER_INCH, FIG_HEIGHT_MM / MM_PER_INCH)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",

    "font.size": 5.4,
    "axes.labelsize": 5.6,
    "axes.titlesize": 5.8,
    "xtick.labelsize": 4.8,
    "ytick.labelsize": 4.8,

    "axes.linewidth": 0.55,
    "xtick.major.width": 0.55,
    "ytick.major.width": 0.55,
    "xtick.major.size": 1.8,
    "ytick.major.size": 1.8,
})

def xi_ratio_and_total(xa, xi, axis):
    xa_sum = xa.sum(axis=axis)
    xi_sum = xi.sum(axis=axis)
    total = xa_sum + xi_sum
    ratio = xi_sum / total.replace(0, np.nan)
    return ratio, total


# ---------- per cell ----------
gt_cell_ratio, gt_cell_total = xi_ratio_and_total(gt["Xa"], gt["Xi"], axis=0)
pred_cell_ratio, pred_cell_total = xi_ratio_and_total(pred["Xa"], pred["Xi"], axis=0)

cell_cmp = pd.DataFrame({
    "groundtruth_xi_ratio": gt_cell_ratio,
    "daisychain_xi_ratio": pred_cell_ratio,
    "groundtruth_classified_reads": gt_cell_total,
    "daisychain_classified_reads": pred_cell_total,
}).dropna()

cell_cmp = cell_cmp[
    (cell_cmp["groundtruth_classified_reads"] >= min_counts)
    & (cell_cmp["daisychain_classified_reads"] >= min_counts)
].copy()

cell_cmp["classified_reads"] = cell_cmp[
    ["groundtruth_classified_reads", "daisychain_classified_reads"]
].mean(axis=1)

cell_cmp["diff"] = cell_cmp["daisychain_xi_ratio"] - cell_cmp["groundtruth_xi_ratio"]
cell_cmp["abs_diff"] = cell_cmp["diff"].abs()


# ---------- per gene ----------
gt_gene_ratio, gt_gene_total = xi_ratio_and_total(gt["Xa"], gt["Xi"], axis=1)
pred_gene_ratio, pred_gene_total = xi_ratio_and_total(pred["Xa"], pred["Xi"], axis=1)

gene_cmp = pd.DataFrame({
    "groundtruth_xi_ratio": gt_gene_ratio,
    "daisychain_xi_ratio": pred_gene_ratio,
    "groundtruth_classified_reads": gt_gene_total,
    "daisychain_classified_reads": pred_gene_total,
}).dropna()

gene_cmp = gene_cmp[
    (gene_cmp["groundtruth_classified_reads"] >= min_counts)
    & (gene_cmp["daisychain_classified_reads"] >= min_counts)
].copy()

gene_cmp["classified_reads"] = gene_cmp[
    ["groundtruth_classified_reads", "daisychain_classified_reads"]
].mean(axis=1)

gene_cmp["diff"] = gene_cmp["daisychain_xi_ratio"] - gene_cmp["groundtruth_xi_ratio"]
gene_cmp["abs_diff"] = gene_cmp["diff"].abs()


# ---------- save filtered tables ----------
cell_cmp.to_csv(outdir / "xi_ratio_per_cell_min10_both.tsv", sep="\t", index=False)
gene_cmp.to_csv(outdir / "xi_ratio_per_gene_min10_both.tsv", sep="\t", index=False)


# ============================================================
# Plot Xi ratio benchmarking
# ============================================================

def xi_ratio_panel(ax, df, title, point_size, show_ylabel=True):
    x = df["groundtruth_xi_ratio"].values
    y = df["daisychain_xi_ratio"].values

    ax.scatter(
        x,
        y,
        s=point_size,
        alpha=0.55,
        color=point_color,
        linewidths=0,
        rasterized=True,
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="black",
        linewidth=0.45,
        alpha=0.6,
        zorder=0,
    )

    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("Groundtruth", labelpad=1.0)

    if show_ylabel:
        ax.set_ylabel("scDaisychain", labelpad=1.0)
    else:
        ax.set_ylabel("")

    ax.set_title(title, fontweight="bold", pad=2)

    # Ensure 0.5 tick is present
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks([0.0, 0.5, 1.0])

    ax.grid(alpha=0.18, linewidth=0.35)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Stats box moved to bottom right
    ax.text(
        0.95,
        0.05,
        f"{corr_text(x, y)}\nn = {len(df):,}",
        transform=ax.transAxes,
        va="bottom",
        ha="right",
        fontsize=4.3,
        bbox=dict(
            boxstyle="round,pad=0.18",
            facecolor="white",
            edgecolor="0.85",
            alpha=0.9,
            linewidth=0.35,
        ),
    )


fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

xi_ratio_panel(
    axes[0],
    cell_cmp,
    title="Per cell Xi Ratio",
    point_size=4.5,
    show_ylabel=True,
)

xi_ratio_panel(
    axes[1],
    gene_cmp,
    title="Per Gene Xi Ratio",
    point_size=10,
    show_ylabel=False,
)

# Transparent background
fig.patch.set_alpha(0)
for ax in fig.axes:
    ax.patch.set_alpha(0)

fig.subplots_adjust(
    left=0.12,
    right=0.985,
    bottom=0.18,
    top=0.90,
    wspace=0.28,
)

savefig("panel_xi_ratio_cell_gene_min10_both_50x50mm")
plt.show()

In [ ]:
# ============================================================
# Gene-level Xi-ratio outliers
# Right panel x-axis fixed from 0 to 0.5
# ============================================================

top_n = 15

top = gene_cmp.sort_values("abs_diff", ascending=False).head(top_n)
top.to_csv(outdir / "top_gene_xi_ratio_outliers.tsv", sep="\t")

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))

# ---------------- left: gene Xi-ratio scatter ----------------
axes[0].scatter(
    gene_cmp["groundtruth_xi_ratio"],
    gene_cmp["daisychain_xi_ratio"],
    c=gene_cmp["classified_reads"],
    s=28,
    alpha=0.85,
    linewidths=0,
    cmap="viridis",
    rasterized=True,
)

axes[0].plot(
    [0, 1],
    [0, 1],
    "--",
    color="black",
    linewidth=0.8,
    alpha=0.7,
)

axes[0].set_xlim(-0.03, 1.03)
axes[0].set_ylim(-0.03, 1.03)
axes[0].set_aspect("equal", adjustable="box")

axes[0].set_xlabel("Ground truth Xi ratio")
axes[0].set_ylabel("Daisychain Xi ratio")
axes[0].set_title("Gene Xi ratios")

# Label top outliers on scatter
for gene in top.index[:8]:
    r = gene_cmp.loc[gene]
    axes[0].text(
        r["groundtruth_xi_ratio"] + 0.01,
        r["daisychain_xi_ratio"] + 0.01,
        str(gene),
        fontsize=7,
    )

# ---------------- right: top absolute errors ----------------
ypos = np.arange(len(top))[::-1]

axes[1].barh(
    ypos,
    top["abs_diff"].values,
    color="#4C78A8",
    edgecolor="black",
    linewidth=0.4,
)

axes[1].set_yticks(ypos)
axes[1].set_yticklabels(top.index)

axes[1].set_xlabel("Absolute Xi-ratio error")
axes[1].set_title("Top gene outliers")

# Extend/fix right plot x-axis from 0 to 0.5
axes[1].set_xlim(0, 0.5)
axes[1].set_xticks(np.arange(0, 0.51, 0.1))

# Add value labels
for y, val in zip(ypos, top["abs_diff"].values):
    axes[1].text(
        val + 0.01,
        y,
        f"{val:.3f}",
        va="center",
        ha="left",
        fontsize=7,
    )

# ---------------- styling ----------------
for ax in axes:
    ax.grid(alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Gene-level Xi-ratio outliers", y=1.05, fontsize=12)

fig.tight_layout()
savefig("panel_gene_xi_ratio_outliers_xlim_0_0p5")

In [ ]:
# ============================================================
# Gene Xi-ratio plot only, highlighting genes with misclassified SNPs
# Requires: gt, pred, outdir, corr_text, savefig already defined
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

min_counts = 10

hap_file = Path(
    "./haplotypes_min10_lc0.01.csv"
)

hap = pd.read_csv(hap_file)

# Find gene column robustly
possible_gene_cols = ["gene", "Gene", "gene_symbol", "gene_name", "GENE"]
gene_col = next((c for c in possible_gene_cols if c in hap.columns), None)

if gene_col is None:
    raise ValueError(
        "Could not find a gene column in haplotype file. "
        f"Available columns are:\n{hap.columns.tolist()}"
    )

# Misclassified SNPs using same accuracy definition as before
hap["accurate"] = hap["REF"].astype(str) == hap["H2"].astype(str)
hap["misclassified"] = ~hap["accurate"]

misclassified_by_gene = (
    hap.groupby(gene_col)["misclassified"]
       .agg(
           n_snps="size",
           n_misclassified_snps="sum"
       )
)

misclassified_by_gene["has_misclassified_snp"] = (
    misclassified_by_gene["n_misclassified_snps"] > 0
)

# ============================================================
# Recompute gene-level Xi ratios, requiring min 10 Xa+Xi counts in both methods
# ============================================================

def xi_ratio_and_total(xa, xi, axis):
    xa_sum = xa.sum(axis=axis)
    xi_sum = xi.sum(axis=axis)
    total = xa_sum + xi_sum
    ratio = xi_sum / total.replace(0, np.nan)
    return ratio, total

gt_gene_ratio, gt_gene_total = xi_ratio_and_total(gt["Xa"], gt["Xi"], axis=1)
pred_gene_ratio, pred_gene_total = xi_ratio_and_total(pred["Xa"], pred["Xi"], axis=1)

gene_cmp_mis = pd.DataFrame({
    "groundtruth_xi_ratio": gt_gene_ratio,
    "daisychain_xi_ratio": pred_gene_ratio,
    "groundtruth_classified_reads": gt_gene_total,
    "daisychain_classified_reads": pred_gene_total,
}).dropna()

gene_cmp_mis = gene_cmp_mis[
    (gene_cmp_mis["groundtruth_classified_reads"] >= min_counts)
    & (gene_cmp_mis["daisychain_classified_reads"] >= min_counts)
].copy()

gene_cmp_mis["classified_reads"] = gene_cmp_mis[
    ["groundtruth_classified_reads", "daisychain_classified_reads"]
].mean(axis=1)

gene_cmp_mis["diff"] = (
    gene_cmp_mis["daisychain_xi_ratio"]
    - gene_cmp_mis["groundtruth_xi_ratio"]
)
gene_cmp_mis["abs_diff"] = gene_cmp_mis["diff"].abs()

# Add misclassified-SNP information
gene_cmp_mis = gene_cmp_mis.join(misclassified_by_gene, how="left")

gene_cmp_mis[["n_snps", "n_misclassified_snps"]] = (
    gene_cmp_mis[["n_snps", "n_misclassified_snps"]]
    .fillna(0)
    .astype(int)
)

gene_cmp_mis["has_misclassified_snp"] = (
    gene_cmp_mis["n_misclassified_snps"] > 0
)

print(
    f"Genes plotted: {len(gene_cmp_mis):,}\n"
    f"Genes with ≥1 misclassified SNP: {gene_cmp_mis['has_misclassified_snp'].sum():,}"
)

display(
    gene_cmp_mis
    .sort_values(["n_misclassified_snps", "abs_diff"], ascending=False)
    .head(20)
)

gene_cmp_mis.to_csv(
    outdir / "gene_xi_ratio_with_misclassified_snps_min10_both.tsv",
    sep="\t"
)

# ============================================================
# Plot: gene Xi ratio only, highlight genes with misclassified SNPs
# ============================================================

fig, ax = plt.subplots(figsize=(4.1, 4.0))

normal = gene_cmp_mis[~gene_cmp_mis["has_misclassified_snp"]]
mis = gene_cmp_mis[gene_cmp_mis["has_misclassified_snp"]]

# Background: genes without misclassified SNPs
ax.scatter(
    normal["groundtruth_xi_ratio"],
    normal["daisychain_xi_ratio"],
    s=24,
    alpha=0.45,
    color="#4C78A8",
    linewidths=0,
    label="No misclassified SNPs",
    rasterized=True,
)

# Highlight: genes with misclassified SNPs
ax.scatter(
    mis["groundtruth_xi_ratio"],
    mis["daisychain_xi_ratio"],
    s=42,
    alpha=0.9,
    color="#E76F51",
    edgecolor="black",
    linewidth=0.35,
    label="≥1 misclassified SNP",
    rasterized=True,
)

# Identity line
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    linewidth=0.9,
    alpha=0.6,
    zorder=0,
)

ax.set_xlim(-0.03, 1.03)
ax.set_ylim(-0.03, 1.03)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("Ground truth Xi ratio")
ax.set_ylabel("Daisychain Xi ratio")
# ax.set_title(
#     # f"Gene Xi-ratio benchmarking\nmin {min_counts} Xa+Xi reads in both methods",
#     pad=10,
# )

ax.text(
    0.05,
    0.95,
    f"{corr_text(gene_cmp_mis['groundtruth_xi_ratio'].values, gene_cmp_mis['daisychain_xi_ratio'].values)}"
    f"\nn = {len(gene_cmp_mis):,}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=8,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="0.85",
        alpha=0.9,
    ),
)

# Label top highlighted outliers
top_label = (
    mis.sort_values(["n_misclassified_snps", "abs_diff"], ascending=False)
       .head(8)
)

for gene, row in top_label.iterrows():
    ax.text(
        row["groundtruth_xi_ratio"] + 0.012,
        row["daisychain_xi_ratio"] + 0.012,
        str(gene),
        fontsize=7,
        color="#7A2E1F",
    )

ax.legend(frameon=False, loc="lower right", fontsize=8)

ax.grid(alpha=0.18)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
savefig("gene_xi_ratio_highlight_misclassified_snps_min10_both")

In [ ]:
# ============================================================
# Check Apoo specifically
# ============================================================

gene_query = "Apoo"

# Find exact case-insensitive match
matches = [
    g for g in common_genes
    if str(g).lower() == gene_query.lower()
]

# If no exact match, show possible partial matches
if len(matches) == 0:
    partial_matches = [
        g for g in common_genes
        if gene_query.lower() in str(g).lower()
    ]
    
    print(f"No exact match for {gene_query!r}.")
    print("Partial matches:")
    display(pd.DataFrame({"matching_gene": partial_matches}))
    
else:
    gene = matches[0]
    print(f"Using gene: {gene}")

    # ---------- total counts ----------
    gt_xa = gt["Xa"].loc[gene].sum()
    gt_xi = gt["Xi"].loc[gene].sum()
    gt_total = gt_xa + gt_xi
    gt_ratio = gt_xi / gt_total if gt_total > 0 else np.nan

    pred_xa = pred["Xa"].loc[gene].sum()
    pred_xi = pred["Xi"].loc[gene].sum()
    pred_total = pred_xa + pred_xi
    pred_ratio = pred_xi / pred_total if pred_total > 0 else np.nan

    apoo_summary = pd.DataFrame({
        "method": ["Ground truth", "Daisychain"],
        "Xa_reads": [gt_xa, pred_xa],
        "Xi_reads": [gt_xi, pred_xi],
        "Xa_plus_Xi_reads": [gt_total, pred_total],
        "Xi_ratio": [gt_ratio, pred_ratio],
    })

    apoo_summary["Xi_ratio_pct"] = apoo_summary["Xi_ratio"] * 100

    display(apoo_summary)

    print(
        f"\nApoo Xi ratio:"
        f"\n  Ground truth: {gt_ratio:.4f} ({gt_ratio*100:.2f}%), n={int(gt_total):,}"
        f"\n  Daisychain:   {pred_ratio:.4f} ({pred_ratio*100:.2f}%), n={int(pred_total):,}"
        f"\n  Difference:   {pred_ratio - gt_ratio:+.4f}"
    )

    # ---------- compare against gene_cmp if present ----------
    if gene in gene_cmp.index:
        print("\nFrom gene_cmp:")
        display(gene_cmp.loc[[gene]])
    else:
        print("\nApoo is not in gene_cmp, probably because it was dropped by filtering/dropna.")

    # ---------- per-cell Apoo counts ----------
    apoo_cells = pd.DataFrame({
        "groundtruth_Xa": gt["Xa"].loc[gene],
        "groundtruth_Xi": gt["Xi"].loc[gene],
        "daisychain_Xa": pred["Xa"].loc[gene],
        "daisychain_Xi": pred["Xi"].loc[gene],
    })

    apoo_cells["groundtruth_total"] = (
        apoo_cells["groundtruth_Xa"] + apoo_cells["groundtruth_Xi"]
    )
    apoo_cells["daisychain_total"] = (
        apoo_cells["daisychain_Xa"] + apoo_cells["daisychain_Xi"]
    )

    apoo_cells["groundtruth_xi_ratio"] = (
        apoo_cells["groundtruth_Xi"] / apoo_cells["groundtruth_total"].replace(0, np.nan)
    )
    apoo_cells["daisychain_xi_ratio"] = (
        apoo_cells["daisychain_Xi"] / apoo_cells["daisychain_total"].replace(0, np.nan)
    )

    # Only cells with Apoo counts in either method
    apoo_cells_nonzero = apoo_cells[
        (apoo_cells["groundtruth_total"] > 0)
        | (apoo_cells["daisychain_total"] > 0)
    ].copy()

    print(f"\nCells with Apoo Xa/Xi counts in either method: {len(apoo_cells_nonzero):,}")
    display(apoo_cells_nonzero.sort_values(
        ["groundtruth_total", "daisychain_total"],
        ascending=False
    ).head(30))

    # Save
    apoo_summary.to_csv(outdir / "Apoo_xi_ratio_summary.tsv", sep="\t", index=False)
    apoo_cells_nonzero.to_csv(outdir / "Apoo_xi_ratio_per_cell.tsv", sep="\t")

    print("\nSaved:")
    print(outdir / "Apoo_xi_ratio_summary.tsv")
    print(outdir / "Apoo_xi_ratio_per_cell.tsv")

In [ ]:
# ============================================================
# Genes with highest Xi ratio
# ============================================================

min_classified_reads = 1   # change to 20/50 if you want stricter filtering
top_n = 30

high_xi = gene_cmp.copy()

# Avoid genes with Xi ratio = 1 from only 1-2 reads
high_xi = high_xi[
    high_xi["classified_reads"] >= min_classified_reads
].copy()

high_xi["mean_xi_ratio"] = high_xi[
    ["groundtruth_xi_ratio", "daisychain_xi_ratio"]
].mean(axis=1)

# Highest Daisychain Xi ratio
top_daisychain_xi = (
    high_xi
    .sort_values(
        ["daisychain_xi_ratio", "daisychain_classified_reads"],
        ascending=[False, False]
    )
    .head(top_n)
)

# Highest ground-truth Xi ratio
top_groundtruth_xi = (
    high_xi
    .sort_values(
        ["groundtruth_xi_ratio", "groundtruth_classified_reads"],
        ascending=[False, False]
    )
    .head(top_n)
)

# Highest average Xi ratio across both methods
top_mean_xi = (
    high_xi
    .sort_values(
        ["mean_xi_ratio", "classified_reads"],
        ascending=[False, False]
    )
    .head(top_n)
)

print("\nTop genes by Daisychain Xi ratio")
display(top_daisychain_xi)

print("\nTop genes by ground-truth Xi ratio")
display(top_groundtruth_xi)

print("\nTop genes by mean Xi ratio")
display(top_mean_xi)

# Save tables
top_daisychain_xi.to_csv(outdir / "top_genes_highest_daisychain_xi_ratio.tsv", sep="\t")
top_groundtruth_xi.to_csv(outdir / "top_genes_highest_groundtruth_xi_ratio.tsv", sep="\t")
top_mean_xi.to_csv(outdir / "top_genes_highest_mean_xi_ratio.tsv", sep="\t")

print("Saved:")
print(outdir / "top_genes_highest_daisychain_xi_ratio.tsv")
print(outdir / "top_genes_highest_groundtruth_xi_ratio.tsv")
print(outdir / "top_genes_highest_mean_xi_ratio.tsv")